# BayesPatch camera-ready experiments (VLM4RWD 2026)

This notebook runs the small additional analyses requested by Reviewer AQ9e and the meta-review.
It reuses the released `bayespatch` package so patching is identical to the paper.

| Part | What it answers | Uses results in paper? |
|---|---|---|
| 0 | Sanity check: reproduces Table 1 forward deltas for L1 center and L11 CLS | No (check only) |
| 1 | Fundus mask: effective in-fundus token counts per region (AQ9e C2) | Yes, after review |
| 2 | Size-matched in-fundus controls for the center region (AQ9e C2) | Yes, after review |
| 3 | Top-k activation-magnitude token patching without region constraint (AQ9e C1) | Yes, after review |
| 4 | Region schematic figure (Reviewer Tgs1) | Yes, after review |
| 5 | Re-run the 528-config sparse grid on the primary 100 pairs (provenance of Fig. 2) | Only after we discuss |
| 6 | Regenerate the four-condition controls incl. DR→DR for primary, random-10, and subsets 1–2 | Only after we discuss |

**Runtime:** Runtime > Change runtime type > T4 GPU, then Runtime > Run all. The patching should take roughly 20 to 30 minutes
(the first APTOS download may add more). At the very end, one zip of all outputs downloads automatically. Upload that zip to Claude.

In [ ]:
# Setup
!pip -q install open_clip_torch datasets
!rm -rf BayesPatch && git clone -q https://github.com/christyzhang2/BayesPatch.git

import sys
sys.path.insert(0, "BayesPatch")

In [ ]:
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

from bayespatch.data import get_image, load_aptos
from bayespatch.model import DEFAULT_PROMPTS, encode_prompts, load_biomedclip
from bayespatch.patching import capture_block_activation, run_patched_margin
from bayespatch.regions import REGIONS, region_token_indices

OUT = Path("camera_ready_outputs")
OUT.mkdir(exist_ok=True)
REPO = Path("BayesPatch")

GRID = 14
N_TOK = 1 + GRID * GRID  # CLS + 196 spatial tokens
ALPHA = 1.0
N_DRAWS = 20             # random draws for in-fundus controls
FUNDUS_PIXEL_THRESHOLD = 0.05   # grayscale intensity in [0, 1] after un-normalising
FUNDUS_TOKEN_FRACTION = 0.5     # a token is "in-fundus" if >= 50% of its 16x16 pixels are fundus

torch.manual_seed(0)
np.random.seed(0)
random.seed(0)

model, preprocess, tokenizer, device = load_biomedclip()
text_features = encode_prompts(model, tokenizer, DEFAULT_PROMPTS, device)
dataset = load_aptos()
print("device:", device, "| prompts:", DEFAULT_PROMPTS)

In [ ]:
# Primary top-10 donors (DR, highest margin) and targets (no-DR, lowest margin),
# read from the released pair-level results so they match Table 1 exactly.
pairs_ref = pd.read_csv(REPO / "results/donor_target_robustness_pair_deltas.csv")
primary = pairs_ref[pairs_ref.pairing_strategy == "top10_extreme"]

fwd_ref = primary[primary.control == "positive_dr_to_no_dr"]
DONORS = sorted(fwd_ref.donor_dataset_index.unique().tolist())    # DR images
TARGETS = sorted(fwd_ref.target_dataset_index.unique().tolist())  # no-DR images
rev_ref = primary[primary.control == "reverse_no_dr_to_dr"]
NODR_CTRL = sorted(rev_ref.donor_dataset_index.unique().tolist())  # no-DR control donors (also reverse donors in the paper)
assert len(DONORS) == 10 and len(TARGETS) == 10 and len(NODR_CTRL) == 10

main_subset = pd.read_csv(REPO / "data/main_subset_indices.csv")


def load_tensor(dataset_index):
    return preprocess(get_image(dataset, dataset_index))


def stack(indices):
    return torch.stack([load_tensor(i) for i in indices]).to(device)


donor_x = stack(DONORS)
target_x = stack(TARGETS)
nodr_ctrl_x = stack(NODR_CTRL)
print("donors:", DONORS)
print("targets:", TARGETS)

In [ ]:
# Batched helpers. Hooks sit on model.visual.trunk.blocks[layer], exactly as in bayespatch/patching.py.
blocks = model.visual.trunk.blocks
N_LAYERS = len(blocks)


@torch.no_grad()
def capture_all_layers(x):
    """Return activations [n_layers, B, 197, D] for a batch."""
    store = [None] * N_LAYERS
    handles = []
    for li, blk in enumerate(blocks):
        def hook(module, inp, out, li=li):
            store[li] = out.detach()
        handles.append(blk.register_forward_hook(hook))
    try:
        model.encode_image(x, normalize=True)
    finally:
        for h in handles:
            h.remove()
    return torch.stack(store)


@torch.no_grad()
def margins(x):
    f = model.encode_image(x, normalize=True)
    s = f @ text_features.T
    return (s[:, 1] - s[:, 0]).float().cpu().numpy()


@torch.no_grad()
def patched_margins(target_batch, donor_act, layer, token_mask, alpha=ALPHA, chunk=50):
    """
    target_batch: [B, 3, 224, 224]; donor_act: [B, 197, D] activations at `layer`;
    token_mask: [B, 197] bool (per-row token sets) or [197] bool (shared set).
    Returns patched margins as numpy [B].
    """
    if token_mask.dim() == 1:
        token_mask = token_mask.unsqueeze(0).expand(target_batch.shape[0], -1)
    out_all = []
    for s in range(0, target_batch.shape[0], chunk):
        tb = target_batch[s:s + chunk]
        da = donor_act[s:s + chunk]
        m = token_mask[s:s + chunk].to(tb.device).unsqueeze(-1).to(da.dtype)

        def hook(module, inp, out):
            return out + alpha * m * (da - out)

        h = blocks[layer].register_forward_hook(hook)
        try:
            out_all.append(margins(tb))
        finally:
            h.remove()
    return np.concatenate(out_all)


def mask_from_indices(indices):
    m = torch.zeros(N_TOK, dtype=torch.bool)
    m[list(indices)] = True
    return m


# Pair layout: row k = (donor k // 10, target k % 10), i.e. every donor x every target.
def pair_batches(donor_set_x, target_set_x):
    nd, nt = donor_set_x.shape[0], target_set_x.shape[0]
    d_idx = torch.arange(nd).repeat_interleave(nt)
    t_idx = torch.arange(nt).repeat(nd)
    return d_idx, t_idx


donor_acts = capture_all_layers(donor_x)    # [L, 10, 197, D]
target_acts = capture_all_layers(target_x)  # [L, 10, 197, D]
nodr_ctrl_acts = capture_all_layers(nodr_ctrl_x)
D_IDX, T_IDX = pair_batches(donor_x, target_x)
pair_targets_x = target_x[T_IDX]
target_base = margins(target_x)
pair_base = target_base[T_IDX.numpy()]


def forward_deltas(layer, token_mask):
    """DR->no-DR deltas for all 100 primary pairs."""
    da = donor_acts[layer][D_IDX]
    return patched_margins(pair_targets_x, da, layer, token_mask) - pair_base


def pairs_frame(deltas, **cols):
    return pd.DataFrame({
        "donor_dataset_index": np.array(DONORS)[D_IDX.numpy()],
        "target_dataset_index": np.array(TARGETS)[T_IDX.numpy()],
        "delta": deltas,
        **cols,
    })

## Part 0: sanity check against the paper
Expected (Table 1, DR→no-DR): L1 center **+0.0995**, L11 CLS **+0.1119**.
If the mean differs by more than ~0.002 or per-pair differences are large, stop and send the output to Claude.

In [ ]:
checks = []
for name, layer, region in [("L1 center", 1, "center"), ("L11 CLS", 11, "cls")]:
    deltas = forward_deltas(layer, mask_from_indices(region_token_indices(region)))
    ref = fwd_ref[fwd_ref.config == name].set_index(["donor_dataset_index", "target_dataset_index"]).delta
    mine = pairs_frame(deltas).set_index(["donor_dataset_index", "target_dataset_index"]).delta
    diff = (mine - ref.reindex(mine.index)).abs()
    checks.append({"config": name, "mean_delta_now": mine.mean(), "mean_delta_paper": ref.mean(),
                   "max_abs_pair_diff": diff.max()})

# Cross-check the batched helper against the released single-pair function for one pair.
one = run_patched_margin(model, target_x[:1], donor_acts[1][:1], text_features, 1, "center", 1.0)
checks.append({"config": "repo function, 1 pair", "mean_delta_now": one - target_base[0],
               "mean_delta_paper": float(fwd_ref[(fwd_ref.config == "L1 center") &
                                                 (fwd_ref.donor_dataset_index == DONORS[0]) &
                                                 (fwd_ref.target_dataset_index == TARGETS[0])].delta.iloc[0]),
               "max_abs_pair_diff": np.nan})
sanity = pd.DataFrame(checks)
sanity.to_csv(OUT / "part0_sanity_check.csv", index=False)
print(sanity.to_string(index=False))

## Part 1: fundus mask and effective token counts per region (AQ9e C2)

In [ ]:
norm = next(t for t in preprocess.transforms if t.__class__.__name__ == "Normalize")
MEAN = torch.tensor(norm.mean).view(3, 1, 1)
STD = torch.tensor(norm.std).view(3, 1, 1)


def fundus_token_mask(x_cpu):
    """x_cpu: [3,224,224] normalised tensor -> (pixel mask [224,224], token mask [197] with CLS False)."""
    img = (x_cpu * STD + MEAN).clamp(0, 1)
    gray = img.mean(0)
    pix = gray > FUNDUS_PIXEL_THRESHOLD
    frac = pix.float().view(GRID, 16, GRID, 16).mean(dim=(1, 3))   # [14,14]
    tok = torch.zeros(N_TOK, dtype=torch.bool)
    tok[1:] = (frac >= FUNDUS_TOKEN_FRACTION).flatten()
    return pix, tok


main_masks = {}
for di in main_subset.dataset_index:
    main_masks[int(di)] = fundus_token_mask(load_tensor(int(di)))[1]

region_rows = []
for region in REGIONS:
    if region == "cls":
        continue
    idx = region_token_indices(region)
    counts = np.array([main_masks[d][idx].sum().item() for d in main_masks])
    region_rows.append({
        "region": region, "nominal_tokens": len(idx),
        "mean_in_fundus_tokens": counts.mean(), "std_in_fundus_tokens": counts.std(),
        "min_in_fundus_tokens": counts.min(), "mean_in_fundus_fraction": counts.mean() / len(idx),
    })
all_counts = np.array([main_masks[d][1:].sum().item() for d in main_masks])
region_rows.append({"region": "all_spatial", "nominal_tokens": 196,
                    "mean_in_fundus_tokens": all_counts.mean(), "std_in_fundus_tokens": all_counts.std(),
                    "min_in_fundus_tokens": all_counts.min(), "mean_in_fundus_fraction": all_counts.mean() / 196})
region_eff = pd.DataFrame(region_rows)
region_eff.to_csv(OUT / "part1_region_effective_tokens.csv", index=False)
print(region_eff.round(3).to_string(index=False))

# QC figure: check the mask looks right on a few images.
qc_ids = list(main_masks)[:8]
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, di in zip(axes.flat, qc_ids):
    x = load_tensor(di)
    img = (x * STD + MEAN).clamp(0, 1).permute(1, 2, 0).numpy()
    tok = main_masks[di][1:].view(GRID, GRID).numpy()
    ax.imshow(img)
    ax.imshow(np.kron(~tok, np.ones((16, 16))), cmap="Reds", alpha=0.35, vmin=0, vmax=1)
    ax.set_title(f"idx {di}: {tok.sum()} in-fundus", fontsize=9)
    ax.axis("off")
plt.suptitle("QC: red = tokens classified as background")
plt.tight_layout()
plt.savefig(OUT / "part1_fundus_mask_qc.png", dpi=150)
plt.show()

## Part 2: size-matched in-fundus controls for the center region (AQ9e C2)
For every primary pair, at every layer, compare the 64-token center block with
(a) 64 random tokens inside the fundus of both images, and
(b) 64 random tokens inside the fundus but outside the center block.
Each control is averaged over 20 random draws. Forward (DR→no-DR) for all layers; reverse at L1.

In [ ]:
center_idx = region_token_indices("center")
center_mask = mask_from_indices(center_idx)
donor_fmask = torch.stack([fundus_token_mask(donor_x[i].cpu())[1] for i in range(10)])
target_fmask = torch.stack([fundus_token_mask(target_x[i].cpu())[1] for i in range(10)])
nodr_ctrl_fmask = torch.stack([fundus_token_mask(nodr_ctrl_x[i].cpu())[1] for i in range(10)])


def sample_masks(allowed, k, rng):
    """allowed: [B,197] bool. Sample k tokens per row (all if fewer). Returns masks, counts."""
    out = torch.zeros_like(allowed)
    counts = []
    for r in range(allowed.shape[0]):
        pool = torch.nonzero(allowed[r]).flatten().tolist()
        pick = pool if len(pool) <= k else rng.sample(pool, k)
        out[r, pick] = True
        counts.append(len(pick))
    return out, np.array(counts)


def control_deltas(layer, d_acts_all, d_idx, t_idx, tx, base, d_fm, t_fm, direction, donor_ids, target_ids):
    rows = []
    both = d_fm[d_idx] & t_fm[t_idx]
    both[:, 0] = False
    ring = both & ~center_mask.unsqueeze(0)
    da = d_acts_all[layer][d_idx]
    dc = patched_margins(tx, da, layer, center_mask) - base
    rows.append(pd.DataFrame({"pair": range(len(dc)), "control": "center_64", "draw": -1,
                              "n_tokens": 64, "delta": dc}))
    for name, allowed in [("in_fundus_random_64", both), ("in_fundus_noncenter_64", ring)]:
        for draw in range(N_DRAWS):
            rng = random.Random(1000 * layer + draw)
            m, cnt = sample_masks(allowed, 64, rng)
            d = patched_margins(tx, da, layer, m) - base
            rows.append(pd.DataFrame({"pair": range(len(d)), "control": name, "draw": draw,
                                      "n_tokens": cnt, "delta": d}))
    df = pd.concat(rows)
    df["layer"] = layer
    df["direction"] = direction
    df["donor_dataset_index"] = df.pair.map(dict(enumerate(np.array(donor_ids)[d_idx.numpy()])))
    df["target_dataset_index"] = df.pair.map(dict(enumerate(np.array(target_ids)[t_idx.numpy()])))
    return df


t0 = time.time()
part2 = []
for layer in range(N_LAYERS):
    part2.append(control_deltas(layer, donor_acts, D_IDX, T_IDX, pair_targets_x, pair_base,
                                donor_fmask, target_fmask, "forward", DONORS, TARGETS))
    print(f"layer {layer} done ({time.time() - t0:.0f}s)")

# Reverse direction at L1, as in the paper: no-DR control donors patched into the top-10 DR images.
R_D, R_T = pair_batches(nodr_ctrl_x, donor_x)
rev_tx = donor_x[R_T]
rev_base = margins(donor_x)[R_T.numpy()]
part2.append(control_deltas(1, nodr_ctrl_acts, R_D, R_T, rev_tx, rev_base, nodr_ctrl_fmask, donor_fmask,
                            "reverse", NODR_CTRL, DONORS))

part2 = pd.concat(part2, ignore_index=True)
part2.to_csv(OUT / "part2_infundus_controls_pairs.csv", index=False)


def target_level(df, value="delta"):
    # average over draws and donors -> one value per target (10 per condition), as in the paper
    return df.groupby(["direction", "layer", "control", "target_dataset_index"])[value].mean().reset_index()


p2_t = target_level(part2)
p2_summary = p2_t.groupby(["direction", "layer", "control"]).delta.agg(["mean", "std", "count"]).reset_index()
p2_summary.to_csv(OUT / "part2_infundus_controls_summary.csv", index=False)

tests = []
for (direction, layer), g in p2_t.groupby(["direction", "layer"]):
    w = g.pivot(index="target_dataset_index", columns="control", values="delta")
    alt = "greater" if direction == "forward" else "less"
    for ctrl in ["in_fundus_random_64", "in_fundus_noncenter_64"]:
        p = wilcoxon(w["center_64"], w[ctrl], alternative=alt).pvalue
        tests.append({"direction": direction, "layer": layer, "comparison": f"center_64 vs {ctrl}",
                      "center_mean": w["center_64"].mean(), "control_mean": w[ctrl].mean(),
                      "alternative": alt, "p_value": p})
p2_tests = pd.DataFrame(tests)
p2_tests.to_csv(OUT / "part2_infundus_controls_tests.csv", index=False)
print(p2_summary[p2_summary.layer.isin([0, 1, 2, 11])].round(4).to_string(index=False))
print(p2_tests[p2_tests.layer == 1].to_string(index=False))

## Part 3: top-k activation-magnitude token patching, no region constraint (AQ9e C1)
At each layer, rank spatial tokens by donor-target activation distance ||A_d − A_t||₂ and patch the top k (k = 49, 64).
Two variants: **pooled** (one token set per layer, averaged over all 100 pairs, like a fixed region) and
**per-pair** (each pair gets its own top-k). Both are selected on the same pairs they are evaluated on,
which favours the magnitude heuristic. Compared against center (64), random-49 (paper's region), and the four quadrants (49).

In [ ]:
part3 = []
token_sets = []
for layer in range(N_LAYERS):
    da = donor_acts[layer][D_IDX]           # [100,197,D]
    ta = target_acts[layer][T_IDX]
    dist = (da - ta).norm(dim=-1).float().cpu()   # [100,197]
    dist[:, 0] = -1.0                              # exclude CLS: spatial tokens only
    pooled = dist.mean(0)

    configs = {}
    for k in (49, 64):
        top_pooled = torch.topk(pooled, k).indices
        configs[f"magnitude_pooled_top{k}"] = mask_from_indices(top_pooled.tolist())
        per_pair = torch.zeros(dist.shape, dtype=torch.bool)
        per_pair.scatter_(1, torch.topk(dist, k, dim=1).indices, True)
        configs[f"magnitude_perpair_top{k}"] = per_pair
        token_sets.append({"layer": layer, "k": k, "pooled_tokens": " ".join(map(str, sorted(top_pooled.tolist()))),
                           "overlap_with_center": len(set(top_pooled.tolist()) & set(center_idx)),
                           "mean_in_fundus_fraction": float(
                               (target_fmask[:, top_pooled].float().mean()))})
    for region in ["center", "random", "q1", "q2", "q3", "q4"]:
        configs[region] = mask_from_indices(region_token_indices(region))

    for name, m in configs.items():
        d = forward_deltas(layer, m)
        part3.append(pairs_frame(d, layer=layer, method=name))
    print(f"layer {layer} done")

part3 = pd.concat(part3, ignore_index=True)
part3.to_csv(OUT / "part3_magnitude_topk_pairs.csv", index=False)
pd.DataFrame(token_sets).to_csv(OUT / "part3_magnitude_token_sets.csv", index=False)

p3_t = part3.groupby(["layer", "method", "target_dataset_index"]).delta.mean().reset_index()
p3_summary = p3_t.groupby(["layer", "method"]).delta.agg(["mean", "std"]).reset_index()
p3_summary.to_csv(OUT / "part3_magnitude_topk_summary.csv", index=False)
table = p3_summary.pivot(index="layer", columns="method", values="mean")
print(table.round(4).to_string())

tests = []
for layer, g in p3_t.groupby("layer"):
    w = g.pivot(index="target_dataset_index", columns="method", values="delta")
    for other in ["magnitude_pooled_top64", "magnitude_perpair_top64"]:
        tests.append({"layer": layer, "comparison": f"center vs {other}",
                      "center_mean": w["center"].mean(), "other_mean": w[other].mean(),
                      "p_center_greater": wilcoxon(w["center"], w[other], alternative="greater").pvalue,
                      "p_center_less": wilcoxon(w["center"], w[other], alternative="less").pvalue})
p3_tests = pd.DataFrame(tests)
p3_tests.to_csv(OUT / "part3_magnitude_topk_tests.csv", index=False)
print(p3_tests.round(4).to_string(index=False))

## Part 4: region schematic figure (Reviewer Tgs1)
One panel per token region on a primary target image; grey cells are tokens the fundus mask marks as background.

In [ ]:
sch_x = target_x[0].cpu()
sch_img = (sch_x * STD + MEAN).clamp(0, 1).permute(1, 2, 0).numpy()
sch_bg = ~fundus_token_mask(sch_x)[1][1:].view(GRID, GRID).numpy()
panels = ["cls", "center", "upper", "lower", "left", "right", "q1", "q2", "q3", "q4", "random"]
titles = {"cls": "CLS (1 token)", "center": "Center (64)", "upper": "Upper (98)", "lower": "Lower (98)",
          "left": "Left (98)", "right": "Right (98)", "q1": "Q1 upper-left (49)", "q2": "Q2 upper-right (49)",
          "q3": "Q3 lower-left (49)", "q4": "Q4 lower-right (49)", "random": "Random (49, seed 0)"}
fig, axes = plt.subplots(2, 6, figsize=(13, 4.8))
for ax, region in zip(axes.flat, panels + ["mask"]):
    ax.imshow(sch_img)
    for g in range(GRID + 1):
        ax.axhline(g * 16 - 0.5, color="white", lw=0.3, alpha=0.5)
        ax.axvline(g * 16 - 0.5, color="white", lw=0.3, alpha=0.5)
    if region == "mask":
        ax.imshow(np.kron(sch_bg, np.ones((16, 16))), cmap="Greys", alpha=0.6, vmin=0, vmax=1)
        ax.set_title("Background tokens (mask)", fontsize=9)
    elif region == "cls":
        ax.text(112, 112, "global\n[CLS] token", ha="center", va="center", color="white", fontsize=10,
                bbox=dict(facecolor="#d95f02", alpha=0.85, edgecolor="none"))
        ax.set_title(titles[region], fontsize=9)
    else:
        sel = np.zeros(GRID * GRID, dtype=float)
        sel[np.array(region_token_indices(region)) - 1] = 1
        ax.imshow(np.kron(sel.reshape(GRID, GRID), np.ones((16, 16))), cmap="Oranges", alpha=0.5, vmin=0, vmax=1)
        ax.set_title(titles[region], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUT / "fig_region_schematic.pdf", bbox_inches="tight")
plt.savefig(OUT / "fig_region_schematic.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Save run metadata and download everything (Parts 0-4).
import open_clip
meta = {"date": time.strftime("%Y-%m-%d %H:%M"), "device": str(device), "torch": torch.__version__,
        "open_clip": open_clip.__version__, "donors": DONORS, "targets": TARGETS, "alpha": ALPHA,
        "n_draws": N_DRAWS, "fundus_pixel_threshold": FUNDUS_PIXEL_THRESHOLD,
        "fundus_token_fraction": FUNDUS_TOKEN_FRACTION}
(OUT / "run_metadata.json").write_text(json.dumps(meta, indent=2))

## Part 5: re-run the full 528-configuration sparse grid on the primary 100 pairs
The released repo has no runnable grid script (`scripts/run_grid.py` only lists the search space), and the
released grid does not record which donor-target pairs it used. This re-run uses the primary top-10 pairs,
forward direction, with the same regions (random = seed 0) and costs as the paper.
**Do not put these numbers in the paper yet** — send them to Claude first so we can decide together.
Takes a few minutes on a T4. Note: for L11 CLS at alpha=1 the patched margin equals the donor's margin, so on these
pairs L11 CLS must come out at about 0.1119 (the paper's grid reports 0.1084, so the original grid used other pairs).

In [ ]:
from bayespatch.objective import patch_cost, sparse_objective

grid_rows = []
for layer in range(N_LAYERS):
    for region in REGIONS:
        m = mask_from_indices(region_token_indices(region, random_seed=0))
        for alpha in (0.25, 0.5, 0.75, 1.0):
            da = donor_acts[layer][D_IDX]
            d = patched_margins(pair_targets_x, da, layer, m, alpha=alpha) - pair_base
            grid_rows.append({"layer": layer, "region": region, "alpha": alpha,
                              "mean_delta": d.mean(), "std_delta": d.std(),
                              "patch_cost": patch_cost(region, alpha),
                              "objective": sparse_objective(d.mean(), region, alpha, lam=0.03)})
    print(f"layer {layer} done")
grid = pd.DataFrame(grid_rows).sort_values("objective", ascending=False)
grid.to_csv(OUT / "part5_grid_rerun_primary_pairs.csv", index=False)
print(grid.head(10).round(4).to_string(index=False))

## Part 6: regenerate the four-condition controls, including DR→DR
Settings: primary (main subset, top-10), random-10 (main subset, random draw with seed 0), and the two independent
subsets (top-10 within each). Selection mirrors `scripts/directionality.py`; the DR→DR control uses DR donors ranked
11–20 by margin (lower margins than the top-10 DR targets), matching the paper's description.
Forward: DR donors → no-DR targets. no-DR control: no-DR donors → no-DR targets.
Reverse: no-DR donors → DR targets. DR control: DR control donors → DR targets.

In [ ]:
from scipy.stats import wilcoxon as _wx

def subset_margins(manifest):
    rows = []
    for r in manifest.itertuples(index=False):
        x = load_tensor(int(r.dataset_index)).unsqueeze(0).to(device)
        rows.append({"dataset_index": int(r.dataset_index), "binary_dr": int(r.binary_dr),
                     "diagnosis": int(r.diagnosis), "margin": float(margins(x)[0])})
    return pd.DataFrame(rows)


def groups_top10(m):
    dr = m[m.binary_dr == 1].sort_values("margin", ascending=False)
    nd = m[m.binary_dr == 0].sort_values("margin", ascending=True)
    return {"dr_donors": dr.dataset_index.iloc[:10].tolist(), "dr_targets": dr.dataset_index.iloc[:10].tolist(),
            "dr_ctrl_donors": dr.dataset_index.iloc[10:20].tolist(),
            "nodr_targets": nd.dataset_index.iloc[:10].tolist(), "nodr_donors": nd.dataset_index.iloc[10:20].tolist()}


def groups_random10(m, seed=0):
    rng = np.random.default_rng(seed)
    dr = rng.permutation(m[m.binary_dr == 1].dataset_index.values).tolist()
    nd = rng.permutation(m[m.binary_dr == 0].dataset_index.values).tolist()
    return {"dr_donors": dr[:10], "dr_targets": dr[:10], "dr_ctrl_donors": dr[10:20],
            "nodr_targets": nd[:10], "nodr_donors": nd[10:20]}


CONFIGS = [("L1 center", 1, "center"), ("L11 CLS", 11, "cls")]
CONDITIONS = [("forward_dr_to_nodr", "dr_donors", "nodr_targets"),
              ("control_nodr_to_nodr", "nodr_donors", "nodr_targets"),
              ("reverse_nodr_to_dr", "nodr_donors", "dr_targets"),
              ("control_dr_to_dr", "dr_ctrl_donors", "dr_targets")]


def run_setting(name, groups):
    imgs = {k: stack(v) for k, v in groups.items()}
    acts = {k: capture_all_layers(x) for k, x in imgs.items()}
    rows = []
    for cfg, layer, region in CONFIGS:
        mask = mask_from_indices(region_token_indices(region))
        for cond, dk, tk in CONDITIONS:
            d_idx, t_idx = pair_batches(imgs[dk], imgs[tk])
            tx = imgs[tk][t_idx]
            base = margins(imgs[tk])[t_idx.numpy()]
            delta = patched_margins(tx, acts[dk][layer][d_idx], layer, mask) - base
            rows.append(pd.DataFrame({"setting": name, "config": cfg, "condition": cond,
                                      "donor_dataset_index": np.array(groups[dk])[d_idx.numpy()],
                                      "target_dataset_index": np.array(groups[tk])[t_idx.numpy()],
                                      "base_margin": base, "delta": delta}))
    return pd.concat(rows)


manifests = {0: main_subset,
             1: pd.read_csv(REPO / "data/seed1_subset_indices.csv"),
             2: pd.read_csv(REPO / "data/seed2_subset_indices.csv")}
margin_tables = {k: subset_margins(v).assign(subset_seed=k) for k, v in manifests.items()}
pd.concat(margin_tables.values()).to_csv(OUT / "part6_subset_margins_all.csv", index=False)

released = pd.read_csv(REPO / "results/subset_margins.csv").set_index("dataset_index").margin
chk = margin_tables[0].set_index("dataset_index").margin
print("max |margin - released margin| on main subset:", float((chk - released.reindex(chk.index)).abs().max()))

settings = {"primary_top10": groups_top10(margin_tables[0]),
            "random10_seed0": groups_random10(margin_tables[0], seed=0),
            "subset1_top10": groups_top10(margin_tables[1]),
            "subset2_top10": groups_top10(margin_tables[2])}
json.dump(settings, open(OUT / "part6_groups.json", "w"), indent=1, default=int)

part6 = pd.concat([run_setting(k, g) for k, g in settings.items()], ignore_index=True)
part6.to_csv(OUT / "part6_four_condition_pairs.csv", index=False)

p6_summary = part6.groupby(["setting", "config", "condition"]).delta.agg(["mean", "std", "count"]).reset_index()
p6_summary.to_csv(OUT / "part6_four_condition_summary.csv", index=False)

tests = []
tl = part6.groupby(["setting", "config", "condition", "target_dataset_index"]).delta.mean().reset_index()
for (setting, cfg), g in tl.groupby(["setting", "config"]):
    v = {c: g[g.condition == c].set_index("target_dataset_index").delta for c in g.condition.unique()}
    tests += [
        {"setting": setting, "config": cfg, "test": "forward > 0",
         "p": _wx(v["forward_dr_to_nodr"], alternative="greater").pvalue},
        {"setting": setting, "config": cfg, "test": "reverse < 0",
         "p": _wx(v["reverse_nodr_to_dr"], alternative="less").pvalue},
        {"setting": setting, "config": cfg, "test": "forward > no-DR control (paired by target)",
         "p": _wx(v["forward_dr_to_nodr"], v["control_nodr_to_nodr"].reindex(v["forward_dr_to_nodr"].index),
                  alternative="greater").pvalue},
        {"setting": setting, "config": cfg, "test": "reverse < DR control (paired by target)",
         "p": _wx(v["reverse_nodr_to_dr"], v["control_dr_to_dr"].reindex(v["reverse_nodr_to_dr"].index),
                  alternative="less").pvalue},
    ]
p6_tests = pd.DataFrame(tests)
p6_tests.to_csv(OUT / "part6_four_condition_tests.csv", index=False)
print(p6_summary.pivot_table(index=["setting", "config"], columns="condition", values="mean").round(4).to_string())
print(p6_tests.to_string(index=False))

In [ ]:
# Download everything
!zip -q -r camera_ready_outputs.zip camera_ready_outputs
from google.colab import files
files.download("camera_ready_outputs.zip")